## Downaload a dataset

In some cases, you need to login to be able to download a dataset. Run this cell, if that is the case.

In [4]:
from huggingface_hub import login
login()

On Huggingface, click on the copy button to get the name of the dataset.

![dataset_from_hugging_face](../images/get_dataset_from_hugging_face.png)

And provide that name to `load_dataset`

In [5]:
from datasets import load_dataset

# Load a specific language subset (recommended — full dataset is huge)
ds = load_dataset("HuggingFaceFW/fineweb-2", name="ita_Latn", split="train", streaming=True)

Resolving data files:   0%|          | 0/85 [00:00<?, ?it/s]

### Extract and save dataset

By Chinchilla's law (20 tokens/param), a ~86M params model needs ~1.72B tokens. At ~4.5 characters per token, that's ~7.74 GB of raw text — I collect 8 GB to leave a small buffer for filtering losses during tokenization.

In [6]:
TARGET_GB      = 8                        # stop after this many GB of text
MIN_CHARS      = 100                      # skip very short documents
MAX_CHARS      = 10_000                   # skip absurdly long ones
OUTPUT_FILE    = "../data/fineweb2_raw.txt"

TARGET_BYTES   = TARGET_GB * 1_000_000_000

In [7]:
total_bytes   = 0
total_docs    = 0
skipped_docs  = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for sample in ds:
        text = sample["text"].replace('\n', ' ')
        length = len(text)

        # Filter by text length
        if length < MIN_CHARS or length > MAX_CHARS:
            skipped_docs += 1
            continue

        f.write(text + '\n')

        total_bytes += len(text.encode("utf-8"))
        total_docs  += 1

        # Progress every 50k docs
        if total_docs % 50_000 == 0:
            print(f"  {total_docs:,} docs | {total_bytes / 1e9:.2f} GB collected")

        if total_bytes >= TARGET_BYTES:
            break

print(f"\nDone!")
print(f"  Documents : {total_docs:,}")
print(f"  Skipped   : {skipped_docs:,}")
print(f"  Size      : {total_bytes / 1e9:.2f} GB")

  50,000 docs | 0.12 GB collected
  100,000 docs | 0.23 GB collected
  150,000 docs | 0.34 GB collected
  200,000 docs | 0.46 GB collected
  250,000 docs | 0.57 GB collected
  300,000 docs | 0.69 GB collected
  350,000 docs | 0.80 GB collected
  400,000 docs | 0.92 GB collected
  450,000 docs | 1.04 GB collected
  500,000 docs | 1.17 GB collected
  550,000 docs | 1.29 GB collected
  600,000 docs | 1.41 GB collected
  650,000 docs | 1.54 GB collected
  700,000 docs | 1.66 GB collected
  750,000 docs | 1.79 GB collected
  800,000 docs | 1.92 GB collected
  850,000 docs | 2.04 GB collected
  900,000 docs | 2.17 GB collected
  950,000 docs | 2.29 GB collected
  1,000,000 docs | 2.42 GB collected
  1,050,000 docs | 2.55 GB collected
  1,100,000 docs | 2.68 GB collected
  1,150,000 docs | 2.80 GB collected
  1,200,000 docs | 2.91 GB collected
  1,250,000 docs | 3.02 GB collected
  1,300,000 docs | 3.13 GB collected
  1,350,000 docs | 3.25 GB collected
  1,400,000 docs | 3.36 GB collected
  1